In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras.datasets import imdb 
from tensorflow.keras.preprocessing import sequence 
from tensorflow.keras.utils import pad_sequences
import sklearn 
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split

In [5]:
# Load word index
word_index = imdb.get_word_index()
# Create reverse word index
reverse_word_index = {value: key for (key, value) in word_index.items()}

In [7]:
model = load_model('imdb_classification.h5')
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (32, 500, 128)              │       1,280,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_1 (SimpleRNN)             │ (32, 128)                   │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (32, 1)                     │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [9]:
#to get an understanding of the weights of the model
model.get_weights()

[array([[-0.19098817, -1.0742145 , -0.8886851 , ...,  0.28324112,
         -0.04167746,  0.9365836 ],
        [-0.02135133, -0.00680458,  0.00165907, ..., -0.02136594,
          0.05161051, -0.03184541],
        [ 0.15487443,  0.06585567, -0.00250226, ..., -0.00296077,
          0.13486806, -0.01631008],
        ...,
        [ 0.04106053, -0.23897259, -0.0992143 , ..., -0.04719506,
          0.02874512, -0.04756314],
        [ 0.03831802,  0.01051176, -0.0179231 , ..., -0.00309769,
          0.06487942, -0.06738427],
        [ 0.12684092, -0.10005921, -0.05943424, ...,  0.02633101,
          0.1374488 , -0.17743157]], dtype=float32),
 array([[-0.12604031, -0.12240301,  0.1168353 , ...,  0.14069732,
         -0.00790822, -0.09169616],
        [-0.10209557,  0.12126009, -0.05250502, ..., -0.10288217,
          0.1636131 ,  0.03399755],
        [ 0.0366761 ,  0.094809  ,  0.04327713, ..., -0.08388648,
         -0.08032281,  0.0544188 ],
        ...,
        [-0.06098272, -0.0624725 , -0.1

In [10]:
## After loading the model, we need to decode the input given to the model 
## decoding the reviews that should be provided to the model

def decode_review(text):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in text])

### preprocess the user input before fitting it in the model
def preprocess_text(text):
    # Encode the text using the word index
    encoded = [word_index.get(word, 2) for word in text.lower().split()]
    
    # Pad the sequence to max length 500
    padded = pad_sequences([encoded], maxlen=500, padding='pre')
    return padded

In [11]:
### prediction fucntion for testing 
def predict_sentiments(review):
    preprocessed_input = preprocess_text(review)
    
    predict = model.predict(preprocessed_input)
    
    sentiment = 'Positive' if predict[0][0] > 0.5 else 'Negative'
    
    return sentiment, predict[0][0]

In [13]:
## example reviews 

example_review = "The movie was fantastic, the acting was great, and the plot was thrilling."
sentiment, score = predict_sentiments(example_review)
print(f"Review: {example_review}\nSentiment: {sentiment}\nPrediction score: {score:.3f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 889ms/step
Review: The movie was fantastic, the acting was great, and the plot was thrilling.
Sentiment: Positive
Prediction score: 0.865
